In [4]:
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from webdriver_manager.chrome import ChromeDriverManager
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from bs4 import BeautifulSoup as bs
from book_dbid import modulebook
import pandas as pd
from translate import kor2eng
import time

options = Options()
options.add_experimental_option("detach", True)
options.add_argument("star-maximized")
# options.add_argument("Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/134.0.0.0 Safari/537.36")
options.add_argument("land=ko_KR")
options.add_argument("Chrome/135.0.0.0")

driver = webdriver.Chrome(
    service=Service(ChromeDriverManager().install()),
    options=options
    )
driver.get("https://search.shopping.naver.com/book/home")

result = []
columns = ['책제목', '상세정보링크', '저자', '출판사', '출간일', '평점', '가격']
    

word = input("검색어: ")

word2 = kor2eng(word)

# 파이썬 검색
search_box = driver.find_element(By.CSS_SELECTOR, '._searchInput_search_text_83jy9._searchInput_placeholder_AG5yA._nlog_click')
search_box.clear()
search_box.send_keys(word)
search_button = driver.find_element(By.CSS_SELECTOR, '._searchInput_button_search_wu9xq._nlog_click')
search_button.click()

for num in range(1,6):
    # 스크롤 후 데이터 수집
    y = 0
    y_step = 1000
    for scroll_times in range(1,8):
        y = y + y_step
        driver.execute_script(f"window.scrollTo(0,{y})")
        time.sleep(0.1)


    for book in driver.find_elements(By.CSS_SELECTOR, 'div.bookListItem_item_inner__edK7P'):
        name = book.find_element(By.CSS_SELECTOR, 'div.bookListItem_title__1mWGq').text

        link = book.find_element(By.CSS_SELECTOR, 'a.bookListItem_info_top__DLxpl').get_attribute('href')

        try:
            author = book.find_elements(By.CSS_SELECTOR, 'span.bookListItem_define_data__fu2A5')[0].text
        except Exception:
            author = ''
        
        try:
            publisher = book.find_elements(By.CSS_SELECTOR, 'span.bookListItem_define_data__fu2A5')[1].text
        except Exception:
            publisher = ''
        elements = book.find_elements(By.CSS_SELECTOR, 'div.bookListItem_detail_date__6_wYJ')
        if elements:
            date = elements[0].text.replace('.','')
        else:
            date = None

        elements = book.find_elements(By.CSS_SELECTOR, 'div.bookListItem_grade__e60mi')
        if elements:
            star_text = elements[0].text
            star = float(star_text[2:6].strip())
        else:
            star = None

        elements = book.find_elements(By.CSS_SELECTOR, 'span.bookPrice_price__Nv4Ee > em')
        if elements:
            price = elements[0].text
            price = price.replace(',','')
        else:
            price = None

        result.append((name, link, author, publisher, date, star, price))
    
    result_df = pd.DataFrame(result, columns = columns)
    
    modulebook(result_df,word2)
    
    paginator = driver.find_element(By.CSS_SELECTOR, ".Paginator_list_paging__XbuO8")
    paginator.find_element(By.LINK_TEXT, f"{num+1}").click()

검색어: 핀테크


In [ ]:
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from webdriver_manager.chrome import ChromeDriverManager
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

options = Options()
options.add_experimental_option("detach", True)
options.add_argument("star-maximized")
# options.add_argument("Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/134.0.0.0 Safari/537.36")
options.add_argument("land=ko_KR")
options.add_argument("Chrome/135.0.0.0")


def kor2eng(keyword):
    driver = webdriver.Chrome(
        service=Service(ChromeDriverManager().install()),
        options=options
        )
    driver.get("https://translate.google.com/?hl=ko&tab=TT&sl=ko&tl=en&op=translate")

    # 파이썬 검색
    search_box = driver.find_element(By.CSS_SELECTOR, 'div.QFw9Te textarea')
    search_box.clear()
    search_box.send_keys(keyword)

    wait = WebDriverWait(driver,10)      # 최대 10초동안 기다려라
    search_text_box = wait.until(EC.presence_of_element_located((By.CSS_SELECTOR, 'div.lRu31')))

    return search_text_box.text

In [9]:
word = '핀테크'
search_box = driver.find_element(By.CSS_SELECTOR, 'div.QFw9Te textarea')
search_box.send_keys(f'{word}')

In [3]:
book.text

'내일의 핀테크\n포인트\n도서\ncoramdeo\n네이버플러스멤버십\n11,700원무료\n신고'